# STT 실습 환경 셋업

이 노트북을 **순서대로** 실행하면 `test.ipynb` 실습 준비가 완료됩니다.

| 단계 | 내용 | 실행 위치 |
|------|------|-----------|
| 1 | uv 설치 | 터미널 |
| 2 | 가상환경 생성 + 패키지 설치 | 터미널 |
| 3 | ffmpeg 설치 | 터미널 |
| 4 | VS Code 커널 전환 | VS Code UI |
| 5 | HuggingFace 토큰 저장 | 이 노트북 |
| 6 | pyannote 약관 동의 | 브라우저 |
| 7 | 모델 다운로드 | 이 노트북 |
| 8 | 설치 검증 | 이 노트북 |

---
## Step 1 — uv 설치 (터미널)

uv는 Python 패키지 관리자입니다. **VS Code 터미널** (`Ctrl+`` `)에서 아래 명령어를 실행하세요.

```powershell
powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"
```

> 설치 후 터미널을 **닫았다가 다시 열어야** `uv` 명령어가 인식됩니다.

In [ ]:
import subprocess

result = subprocess.run(["uv", "--version"], capture_output=True, text=True)
if result.returncode == 0:
    print(f"✅ uv 설치 확인: {result.stdout.strip()}")
else:
    print("❌ uv가 인식되지 않습니다.")
    print("   → Step 1의 터미널 명령어를 실행한 뒤 터미널을 재시작하세요.")

---
## Step 2 — Python 3.12 설치 + 가상환경 생성 + 패키지 설치 (터미널)

터미널에서 아래 명령어를 **순서대로** 실행하세요.

```powershell
# 1) Python 3.12 설치 (이미 설치되어 있으면 건너뜁니다)
uv python install 3.12

# 2) Python 3.12 가상환경 생성
uv venv STT_env --python 3.12

# 3) 가상환경 활성화
STT_env\Scripts\activate

# 4) 패키지 설치 (수 분 소요)
uv pip install -r requirements.txt
```

> `requirements.txt`가 없는 경우 강사에게 파일을 받으세요.

In [ ]:
from pathlib import Path

venv_python = Path.cwd() / "STT_env" / "Scripts" / "python.exe"

if venv_python.exists():
    result = subprocess.run([str(venv_python), "--version"], capture_output=True, text=True)
    print(f"✅ 가상환경 확인: {venv_python}")
    print(f"   Python 버전: {result.stdout.strip()}")
else:
    print("❌ 가상환경이 없습니다.")
    print("   → Step 2의 터미널 명령어를 실행하세요.")

---
## Step 3 — ffmpeg 설치 (터미널)

ffmpeg는 mp3 등 오디오 파일 디코딩(librosa)에 필요합니다.

```powershell
winget install ffmpeg
```

> 설치 후 **VS Code를 완전히 종료하고 재시작**해야 ffmpeg가 인식됩니다.

In [ ]:
result = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)
if result.returncode == 0:
    first_line = result.stdout.splitlines()[0]
    print(f"✅ ffmpeg 설치 확인: {first_line}")
else:
    print("❌ ffmpeg가 인식되지 않습니다.")
    print("   → Step 3의 터미널 명령어를 실행한 뒤 VS Code를 재시작하세요.")

---
## Step 4 — VS Code 커널 전환

패키지가 설치된 `STT_env` 환경을 이 노트북의 커널로 지정해야 합니다.

1. 우측 상단 **커널 선택** 버튼 클릭 (또는 `Ctrl+Shift+P` → `Notebook: Select Kernel`)
2. **Python Environments...** 선택
3. **`STT_env`** 선택

> 커널 전환 후 이 셀 아래부터 다시 실행하세요.

---
## Step 5 — HuggingFace 토큰 발급 및 저장

화자 분리 모델(pyannote)은 HuggingFace 로그인이 필요합니다.

### 토큰 발급 방법
1. [https://huggingface.co](https://huggingface.co) 에서 회원가입 또는 로그인
2. 우측 상단 프로필 → **Settings** → **Access Tokens**
3. **New token** 클릭 → 이름 입력 → **Read** 권한 선택 → **Generate**
4. 생성된 토큰(`hf_...`)을 복사

아래 셀을 실행하면 토큰 입력창이 나타납니다.

In [ ]:
import json
from pathlib import Path

keys_file = Path.cwd() / "keys.json"

if keys_file.exists():
    print("✅ keys.json 이미 존재합니다. 토큰 입력을 건너뜁니다.")
    with open(keys_file) as f:
        keys = json.load(f)
    token_preview = keys.get("hf_token", "")[:8] + "..."
    print(f"   저장된 토큰: {token_preview}")
else:
    hf_token = input("HuggingFace 토큰을 입력하세요 (hf_...): ").strip()
    if not hf_token.startswith("hf_"):
        print("⚠️  토큰이 'hf_'로 시작하지 않습니다. 올바른 토큰인지 확인하세요.")
    keys = {"hf_token": hf_token}
    keys_file.write_text(json.dumps(keys, indent=2), encoding="utf-8")
    print(f"✅ keys.json 저장 완료: {keys_file}")

---
## Step 6 — pyannote 모델 약관 동의 (브라우저)

pyannote 모델은 HuggingFace에서 **약관 동의 후** 다운로드할 수 있습니다.  
아래 두 페이지에서 각각 **"Agree and access repository"** 버튼을 클릭하세요.

- [https://huggingface.co/pyannote/speaker-diarization-3.1](https://huggingface.co/pyannote/speaker-diarization-3.1)
- [https://huggingface.co/pyannote/segmentation-3.0](https://huggingface.co/pyannote/segmentation-3.0)

> 로그인된 상태에서 진행해야 합니다.

---
## Step 7 — 모델 다운로드

아래 셀을 실행하면 4개 모델을 `models/` 폴더에 다운로드합니다.  
**총 용량 약 6~8 GB**, 인터넷 속도에 따라 수십 분 소요될 수 있습니다.

| 모델 | 용량 (약) | 용도 |
|------|-----------|------|
| Qwen/Qwen3-ASR-1.7B | 3.5 GB | 음성 → 텍스트 (Qwen3) |
| pyannote/speaker-diarization-3.1 | 수백 MB | 화자 분리 |
| pyannote/segmentation-3.0 | 수백 MB | 음성 구간 감지 |

In [ ]:
import json
from pathlib import Path
from huggingface_hub import snapshot_download, login

# 토큰 로드 및 로그인
with open("keys.json") as f:
    keys = json.load(f)

login(token=keys["hf_token"], add_to_git_credential=False)
print("✅ HuggingFace 로그인 완료\n")

# 다운로드할 모델 목록: (repo_id, local_dir, 존재 확인용 파일/폴더)
MODELS = [
    (
        "Qwen/Qwen3-ASR-1.7B",
        "models/Qwen3-ASR",
        "models/Qwen3-ASR/config.json",
    ),
    (
        "pyannote/speaker-diarization-3.1",
        "models/pyannote-diarization",
        "models/pyannote-diarization/config.yaml",
    ),
    (
        "pyannote/segmentation-3.0",
        "models/pyannote-segmentation",
        "models/pyannote-segmentation/config.yaml",
    ),
]

base = Path.cwd()

for repo_id, local_dir, check_path in MODELS:
    check = base / check_path
    target = base / local_dir
    if check.exists():
        print(f"✅ {repo_id}  →  이미 다운로드됨")
    else:
        print(f"⬇️  {repo_id} 다운로드 중...")
        snapshot_download(repo_id=repo_id, local_dir=str(target))
        print(f"✅ {repo_id}  →  완료")

print("\n모든 모델 다운로드 완료!")

---
## Step 8 — 설치 검증

In [ ]:
import sys
from pathlib import Path

print("=== 환경 검증 ===")
print(f"Python: {sys.version}")

# 패키지 임포트 확인
checks = [
    ("torch",                 "PyTorch"),
    ("transformers",          "Transformers"),
    ("qwen_asr",              "Qwen ASR"),
    ("pyannote.audio",        "pyannote.audio"),
    ("soundfile",             "soundfile"),
    ("librosa",               "librosa"),
    ("huggingface_hub",       "huggingface_hub"),
]

print("\n--- 패키지 ---")
all_ok = True
for module, name in checks:
    try:
        __import__(module)
        print(f"  ✅ {name}")
    except ImportError:
        print(f"  ❌ {name}  ← 설치 필요")
        all_ok = False

# GPU 확인
import torch
print("\n--- 디바이스 ---")
if torch.cuda.is_available():
    print(f"  ✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"     VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("  ⚠️  GPU 없음 — CPU로 실행됩니다 (추론 속도가 느립니다)")

# 모델 파일 확인
print("\n--- 모델 파일 ---")
model_checks = [
    ("models/Qwen3-ASR/config.json",              "Qwen3-ASR-1.7B"),
    ("models/pyannote-diarization/config.yaml",   "pyannote diarization"),
    ("models/pyannote-segmentation/config.yaml",  "pyannote segmentation"),
]
for rel_path, name in model_checks:
    p = Path.cwd() / rel_path
    if p.exists():
        print(f"  ✅ {name}")
    else:
        print(f"  ❌ {name}  ← Step 7을 다시 실행하세요")
        all_ok = False

# ffmpeg 확인
import subprocess
print("\n--- ffmpeg ---")
r = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)
if r.returncode == 0:
    print(f"  ✅ {r.stdout.splitlines()[0]}")
else:
    print("  ❌ ffmpeg 없음  ← Step 3을 다시 실행하세요")
    all_ok = False

print()
if all_ok:
    print("🎉 모든 항목 확인 완료! test.ipynb를 실행할 준비가 되었습니다.")
else:
    print("⚠️  일부 항목이 누락되었습니다. 위의 ❌ 항목을 확인하세요.")